# 12 — Bonus: under the LLM layer

**What you'll learn**

- What actually crosses the wire: the exact messages `ChatAdapter` builds from a signature — and why the `->` never reaches the model.
- Why models follow the format at all, and the **fallback ladder** DSPy climbs when they don't — read straight from the installed source.
- The `response_format` protocol: JSON mode vs strict structured outputs, and the literal schema payload DSPy builds from your signature.
- How providers *enforce* a schema during decoding — constrained decoding, token by token — and the edges of that guarantee.

*Time: ~30 min. Cost: < $0.01 — two tiny LM calls; every other cell runs offline against the installed libraries.*

*This is bonus material — nothing in the course depends on it. It answers the question notebooks 01–02 leave open: a signature is a promise about format, and a model is a text generator with no obligation to keep promises. What, exactly, makes it keep them?*

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import dspy
from dotenv import load_dotenv              # pip install -r requirements.txt if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead
# STRONG_MODEL = "openrouter/moonshotai/kimi-k2.6"
# STRONG_MODEL = "openrouter/openai/gpt-4o"

lm = dspy.LM(MODEL, max_tokens=1000)        # cache=True by default: reruns are ~free
dspy.configure(lm=lm)

### Phoenix observability (optional)

Run the next cell to stream this notebook's LM calls into a local [Phoenix](https://phoenix.arize.com) trace UI — or skip it; nothing below depends on tracing. It finds or starts the Phoenix server, prints the link, and names the session by date (`dspy-lab-YYYY-MM-DD`) so different days never mix. Notebook 11 tells the full story.

In [ ]:
# optional: Phoenix tracing (see notebook 11)
import obs

obs.enable_phoenix()

### MLflow observability (optional)

Run the next cell to record this notebook's LM calls with MLflow instead (or as well): logs land in a repo-local store that is safe against remnants and reruns, and the second line finds or starts the MLflow UI — click the printed link to browse them. Equally optional; equally explained in notebook 11.

In [ ]:
# optional: MLflow tracing (see notebook 11)
import obs

obs.enable_mlflow()
obs.mlflow_ui()            # prints the link where the traces appear

## Three mechanisms, in increasing order of force

Everything between your program and the model is format negotiation, and it happens at three levels:

1. **Ask nicely, in-band.** The prompt itself spells out the format — `ChatAdapter`'s field markers, `JSONAdapter`'s field list. Nothing is guaranteed; compliance is *checked* on the way back.
2. **Ask the server, out-of-band.** A `response_format` parameter rides in the request body next to `messages`. The provider guarantees syntactically valid JSON (*JSON mode*) or conformance to your exact schema (*structured outputs*).
3. **Make invalid output unrepresentable.** The strongest form of 2: the server enforces the schema *during decoding*, masking every token that would break it.

DSPy uses all three as a ladder, preferring the friendly end and escalating on failure. This notebook climbs it from the bottom.

## Rung 0: what the model actually sees

Notebook 02 showed a render; look again with fresh eyes. We use `dspy.ChainOfThought`, so the signature carries a `reasoning` output field ahead of `answer` — remember that ordering; it becomes load-bearing at the bottom of the notebook. No LM call here: `format()` is pure string-building.

In [ ]:
class OneWordQA(dspy.Signature):
    """Answer in exactly one word."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="one word only")

cot = dspy.ChainOfThought(OneWordQA)          # same signature + a `reasoning` output field

messages = dspy.ChatAdapter().format(
    cot.predict.signature, demos=[], inputs={"question": "What is the capital of France?"}
)
for m in messages:
    print("=" * 26, m["role"].upper(), "=" * 26)
    print(m["content"])

> **What you should see:** a system message that enumerates your fields, lays out a fill-in template of `[[ ## reasoning ## ]]` / `[[ ## answer ## ]]` markers, and quotes your docstring as "your objective" — then a user message carrying the input and a closing reminder of exactly which marker to start with.
>
> Notice what is *absent*: the arrow. Signatures compile away entirely — the model's task is "follow explicit formatting instructions written in plain English", nothing DSPy-specific.

## Why models comply (mostly)

No mainstream model is pretrained or finetuned on DSPy's markers. They work because of three ordinary facts:

- **Format-following is a trained skill.** Chat models are explicitly post-trained (SFT and RLHF) on instruction-following that includes format compliance — "respond using exactly this template", JSON output, structured extraction. DSPy rides that general capability, not any specific syntax.
- **The markers are chosen for the parser, not the model.** `[[ ## answer ## ]]` is deliberately weird: near-zero collision with natural text, trivial to find with a regex. Any equally strange convention would do.
- **Demos double as format lessons.** Every few-shot demo an optimizer attaches (notebook 02) is rendered in the same marker format — the prompt doesn't just describe the format, it demonstrates it.

And a fourth fact, which motivates everything below: compliance is **checked, never assumed**. `Adapter.parse` regexes the markers back out; a miss raises `AdapterParseError`. "Mostly" is doing real work in this heading — the rest of the notebook is about the miss.

In [ ]:
pred = cot(question="What is the capital of France?")
dspy.inspect_history(n=1)

> **What you should see:** the same render you built by hand, now followed by the model's reply — reasoning and answer between the exact markers, closed with `[[ ## completed ## ]]`. One request, one reply, one regex parse: rung 0 working as designed.

## The fallback ladder, read from the source

What happens when the reply *doesn't* parse is not in the docs — it's a few lines into the default adapter's own code, which we can read right here:

In [ ]:
import inspect

from dspy.adapters.chat_adapter import ChatAdapter

print(inspect.getsource(ChatAdapter.__call__))

> **What you should see:** a `try/except` around the normal call. On failure, `ChatAdapter` *re-runs the entire call through a `JSONAdapter`* — unless the failure was an `LMError` (a network or provider failure would look exactly the same in a different format, so it propagates), the adapter already *is* a `JSONAdapter` (no infinite ladder), or you opted out with `ChatAdapter(use_json_adapter_fallback=False)`.

Spelled out, the whole ladder is:

1. **`ChatAdapter`** — markers, requested in-band, regex-parsed. A parse failure falls through to...
2. **`JSONAdapter`, strict rung** — if the provider can enforce schemas, DSPy builds one from your signature's output fields and sends it; the server guarantees the reply matches it. If the provider can't — or the signature can't be expressed as a strict schema (an open-ended `dict[str, Any]` output field, for instance) — it steps down to...
3. **`JSONAdapter`, JSON-mode rung** — `{"type": "json_object"}`: the server guarantees valid JSON *syntax*; your fields are still requested in-band; and a lenient parser (`json_repair`) absorbs markdown fences, trailing commas, and similar damage.
4. **Plain prompting** — providers that don't accept `response_format` at all simply get asked, with the same lenient parsing.

Below `JSONAdapter` there is no rung: an error there finally surfaces. Two adapters sit *off* the ladder for special situations — `XMLAdapter` (some small models hold XML tags better than JSON) and `TwoStepAdapter` (a strong model answers free-form, then a small extraction model fills your fields — for models too weak for any format discipline).

## Which rung does *your* model land on?

DSPy decides per model, statically, from LiteLLM's capability map — no probe request is sent, so constructing LMs here is free:

In [ ]:
for name in [MODEL, STRONG_MODEL, "openai/gpt-4o-mini", "ollama_chat/llama3.2"]:
    lm = dspy.LM(name)
    accepts = "response_format" in lm.supported_params
    print(f"{name:42s} response_format: {str(accepts):5s} schema enforcement: {lm.supports_response_schema}")

> **What you should see** (values come from the installed LiteLLM's model map, so they can drift as it updates): the course defaults — DeepSeek via OpenRouter — accept `response_format` but **not** schema enforcement, so their fallback rung is JSON mode. `gpt-4o-mini` supports the full strict rung.
>
> Worth internalizing: the ladder's shape depends on the model you set in `.env`. The map decides, not hope.

## The protocol: one field in the request body

There is no special channel. The same HTTPS chat-completions request that carries `messages` grows one more field. JSON mode is the weak form:

```json
{"model": "...", "messages": [...], "response_format": {"type": "json_object"}}
```

The server promises valid JSON — but knows nothing about your fields, which still travel in-band in the prompt. (One protocol quirk worth knowing: OpenAI's JSON mode refuses to run unless the word "JSON" appears somewhere in your messages — a guard against prompts that force a format they never mention.)

Structured outputs is the strong form — your schema itself goes to the server:

```json
{"response_format": {"type": "json_schema", "json_schema": {
    "name": "DSPyProgramOutputs", "strict": true,
    "schema": {"type": "object",
               "properties": {"reasoning": {"type": "string"}, "answer": {"type": "string"}},
               "required": ["reasoning", "answer"], "additionalProperties": false}}}}
```

Strict mode accepts only a subset of JSON Schema: every property listed in `required`, `additionalProperties: false` at every level, no open-ended maps. That subset is exactly why DSPy post-processes the schema it generates — and why a `dict[str, Any]` output field forces the JSON-mode rung. Two more protocol notes:

- **LiteLLM translates per provider.** OpenAI and OpenRouter take this shape as-is; Gemini has its own `response_schema` field; providers with no native support can have it emulated as a forced tool call whose parameter schema is yours.
- **Tool calling is the sibling protocol.** The schemas behind tools (notebook 08) travel in `tools[].function.parameters` — same JSON Schema, same server-side machinery.

DSPy builds that schema from your signature via a pydantic detour: output fields become a pydantic model, the model emits JSON Schema, and a post-pass enforces the strict subset. The builder is private API — pinned here to dspy 3.3.0 — which makes it perfect for inspection:

In [ ]:
import json

from dspy.adapters.json_adapter import _get_structured_outputs_response_format

schema_model = _get_structured_outputs_response_format(cot.predict.signature)
print(json.dumps(schema_model.model_json_schema(), indent=2))

print()
print(dspy.JSONAdapter().format(
    cot.predict.signature, demos=[], inputs={"question": "Capital of France?"}
)[-1]["content"])

> **What you should see:** the `DSPyProgramOutputs` schema — `reasoning` and `answer` both present, both `required`, `additionalProperties: false` — then `JSONAdapter`'s user message, which ends by asking for the fields *in order*: a JSON object with `reasoning` first, `answer` second.
>
> Belt and braces, by design: the schema constrains the server while the prompt still teaches the model — necessary on the JSON-mode rung, harmless on the strict one.

In [ ]:
with dspy.context(adapter=dspy.JSONAdapter()):
    pred = cot(question="What is the capital of France?")

dspy.inspect_history(n=1)
print(pred.answer)

> **What you should see:** the JSON spelling of the same task, and a raw reply that is a JSON object — schema-enforced by the server if your model sits on the strict rung, prompt-guided otherwise. Same signature, same `Prediction` out the other end; only the wire format changed. (`dspy.context` restores `ChatAdapter` on exit.)

## How the strict rung is enforced: constrained decoding

A model emits one token at a time by sampling from a probability distribution over its whole vocabulary. Schema enforcement happens *inside that loop*:

1. The server compiles your JSON Schema into a grammar — a context-free grammar, or a regex-derived finite-state machine.
2. Before sampling **each token**, it asks the compiled automaton which tokens can legally extend the output so far.
3. Every illegal token's logit is set to negative infinity; the distribution is renormalized; sampling proceeds.

Invalid output is not detected and retried — it is **unrepresentable**. The engineering wrinkle is that token boundaries don't align with grammar symbols (`{"reasoning": "` might be one token or five), so engines precompute token-to-grammar-state transition tables. Compilation is cached per schema, which is why the *first* request with a new schema pays extra latency and later ones don't.

The mechanism is public in the open-source engines — Outlines, XGrammar, llguidance, llama.cpp's GBNF grammars, and the "guided decoding" backends in vLLM and SGLang all implement this same idea — and OpenAI has described its implementation in the same terms. Like notebook 11, this section cites no papers: it is engineering, and the engines named above are readable primary sources.

## The edges of the guarantee

Three boundaries keep the strict rung honest:

- **Structure, never content.** Enforcement guarantees the *shape*; a schema-perfect hallucination is still a hallucination. This is why the course spent notebook 04 on metrics — no decoder mask can check that an answer is *right*.
- **Refusals need a side channel.** Under a forced grammar, a model declining a request would have to phrase the refusal *inside* your schema fields. OpenAI's structured outputs carve out a separate `refusal` field in the response for exactly this reason.
- **Force has a price.** Masking can push the model off its natural distribution mid-generation, and heavily constrained decoding measurably degrades output quality on some tasks. That trade-off is a large part of why DSPy's *default* remains the free-text marker format — fields emitted in signature order, `reasoning` before `answer`, the model thinking in its own voice — with enforcement kept as the safety net rather than the first resort.

And one full-circle footnote: there is exactly one case where a model *is* literally trained on this syntax — `BootstrapFinetune` (notebook 09) finetunes weights on adapter-formatted traces, making the format in-distribution by construction. The ladder, climbed from the other end.

## Recap

| Layer | Mechanism | Guarantee |
|---|---|---|
| in-band spec | markers / field lists in the prompt; `Adapter.parse` checks, `AdapterParseError` on a miss | none — verified after the fact |
| JSON mode | `response_format: {"type": "json_object"}` | valid JSON syntax only |
| structured outputs | `{"type": "json_schema", "strict": true}` + your schema | exact schema conformance |
| constrained decoding | schema -> grammar -> per-token logit masking | how the strict rung is implemented |

| Piece | One-liner |
|---|---|
| the ladder | `ChatAdapter` -> `JSONAdapter` (strict rung -> JSON-mode rung) -> the error finally surfaces |
| `use_json_adapter_fallback=False` | opt out of the ladder |
| capability probe | `lm.supported_params` / `lm.supports_response_schema` — LiteLLM's static map picks the rung |
| `json_repair` | lenient parsing on every JSON rung |
| `XMLAdapter` / `TwoStepAdapter` | off-ladder options: XML-happy models / answer-then-extract for weak ones |
| the edge | structure is guaranteed, content never — metrics stay load-bearing |

## Exercises

1. Run the capability probe on providers you actually use — a Gemini model, a local Ollama tag, an Anthropic model — and predict each one's rung *before* printing it. The raw map is one import away: `litellm.get_supported_openai_params(model=...)` and `litellm.supports_response_schema(...)`.
2. Render `cot.predict.signature` through `dspy.XMLAdapter().format(...)` and diff it against the ChatAdapter render at the top of this notebook. Which would you bet a 3-billion-parameter model follows more reliably, and why?
3. Print `inspect.getsource(JSONAdapter.__call__)` (import it from `dspy.adapters.json_adapter`) and its helper `_json_adapter_call_common`, and find the gates that force the JSON-mode rung even on schema-capable providers. Why can an open-ended `dict[str, Any]` output field never ride the strict rung? The answer is in the strict subset's rules.

That's the whole stack: from `"question -> answer"` down to a masked logit.